# Smoking is bad

In [ ]:
# TODO: install all the necessary packages

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
data_path = Path.cwd() / "data" / "smoking_data.csv"
df = pd.read_csv(data_path)

display(df.head())

In [ ]:
# TODO: what are the dimensions of the data frame? - 1314 * 6
print("Rows and columns:", df.shape)
df.shape

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?
# String columns are not included in the analysis as by default, describe() summarises the numerical columns.

df.describe()


In [ ]:
# TODO: do not forget to check the null values
df.isnull().sum()

In [ ]:
# clean data frame and make easier to work with
df_clean = df.copy()

df_clean["gender_num"] = df_clean["gender"].map({"female": 0, "male": 1})
df_clean["smoker_num"] = df_clean["smoker"].map({"No": 0, "Yes": 1})
df_clean["outcome_num"] = df_clean["outcome"].map({"Dead": 0, "Alive": 1})

display(
    df_clean[["outcome_num", "age", "gender_num", "smoker_num"]].describe()
)

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.
df_clean.groupby(["smoker_num"]).agg(prob=("outcome_num", np.mean))

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
sns.boxplot(data=df, x="smoker", y="age", order=["No", "Yes"])

In [ ]:
df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=[0, 30, 40, 50, 60, 70, float("inf")],
    labels=["<30", "30–39", "40–49", "50–59", "60–69", "70+"],
    right=False,
)

age_summary = (
    df_clean.groupby(["age_group", "smoker_num"], observed=True)
    .agg(
        survival_rate=("outcome_num", "mean"),
        count=("outcome_num", "count"),
    )
    .reset_index()
)

display(age_summary)

In [ ]:
sns.barplot(
    data=age_summary,
    x="age_group",
    y="survival_rate",
    hue="smoker_num",
    hue_order=[0, 1],
    errorbar=None,
)

plt.title("Survival Proportion by Age Group and Smoking Status")
plt.xlabel("Age Group")
plt.ylabel("Survival Proportion")
plt.ylim(0, 1)
plt.legend(title="Smoker", labels=["No", "Yes"])
plt.tight_layout()
plt.show()

In [ ]:
## Interpretation

The overall survival proportion is higher among smokers in this dataset.
However, this does not establish that smoking increases life expectancy.

Age is a potential confounding variable because it may be associated
with both smoking status and survival. Comparing the age distributions
and survival proportions within age groups helps investigate this.

The age-group comparisons should be interpreted alongside the sample
sizes. Small groups can produce unstable estimates, and broad age bands
may leave residual differences in age.

These observational comparisons alone cannot establish a causal effect.